**Create the Bronze notebook**

In [4]:
# ============================================================
# Bronze Layer Ingestion Notebook
# Reads all raw sources from Lakehouse Files and writes them
# as Bronze Delta tables with standardized audit columns.
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, BooleanType
from datetime import datetime

BRONZE_INGESTION_TS = datetime.utcnow().isoformat()
SOURCE_SYSTEM_FILES = "manual_upload_v1"  # per ADR-003


def add_audit_columns(df, source_name: str):
    """Standard audit columns applied to every Bronze table for lineage and debugging."""
    return (
        df.withColumn("_bronze_ingested_at", F.lit(BRONZE_INGESTION_TS).cast(TimestampType()))
          .withColumn("_source_system", F.lit(source_name))
          .withColumn("_source_file", F.input_file_name())
    )


print(f"Bronze ingestion run started at {BRONZE_INGESTION_TS}")

StatementMeta(, 6f7de58d-2e89-4403-9885-2359387de05c, 6, Finished, Available, Finished, False)

Bronze ingestion run started at 2026-08-26T13:37:39.757246


**Postgres-sourced tables**

In [5]:
# ---- Customers ----
customers_raw = spark.read.option("header", True).csv("Files/raw/postgres/customers.csv")
customers_bronze = add_audit_columns(customers_raw, "postgres_customers")
customers_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_customers")
print(f"bronze_customers: {customers_bronze.count()} rows")

# ---- Products ----
products_raw = spark.read.option("header", True).csv("Files/raw/postgres/products.csv")
products_bronze = add_audit_columns(products_raw, "postgres_products")
products_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_products")
print(f"bronze_products: {products_bronze.count()} rows")

# ---- Orders ----
orders_raw = spark.read.option("header", True).csv("Files/raw/postgres/orders.csv")
orders_bronze = add_audit_columns(orders_raw, "postgres_orders")
orders_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_orders")
print(f"bronze_orders: {orders_bronze.count()} rows")

# ---- Order Items ----
order_items_raw = spark.read.option("header", True).csv("Files/raw/postgres/order_items.csv")
order_items_bronze = add_audit_columns(order_items_raw, "postgres_order_items")
order_items_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_order_items")
print(f"bronze_order_items: {order_items_bronze.count()} rows")

StatementMeta(, 6f7de58d-2e89-4403-9885-2359387de05c, 7, Finished, Available, Finished, False)

bronze_customers: 510 rows
bronze_products: 100 rows
bronze_orders: 2000 rows
bronze_order_items: 6024 rows


— CSV loyalty source

In [6]:
loyalty_raw = spark.read.option("header", True).csv("Files/raw/loyalty/loyalty_export.csv")
loyalty_bronze = add_audit_columns(loyalty_raw, "csv_loyalty_export")
loyalty_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_loyalty")
print(f"bronze_loyalty: {loyalty_bronze.count()} rows")

StatementMeta(, 6f7de58d-2e89-4403-9885-2359387de05c, 8, Finished, Available, Finished, False)

bronze_loyalty: 305 rows


**JSON reviews source (nested — read differently from flat CSVs)**

In [7]:
# multiLine=True is required because the JSON is a single pretty-printed
# object, not newline-delimited JSON (NDJSON)
reviews_raw = spark.read.option("multiLine", True).json("Files/raw/reviews/product_reviews.json")

# The actual reviews are nested inside a top-level "reviews" array — explode it
# into one row per review before flattening further in Silver (Phase 6).
reviews_exploded = reviews_raw.select(
    F.col("source").alias("_feed_source"),
    F.col("generated_at_utc"),
    F.explode("reviews").alias("review")
)

reviews_bronze = add_audit_columns(reviews_exploded, "json_reviews_feed")
reviews_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_reviews")
print(f"bronze_reviews: {reviews_bronze.count()} rows")
reviews_bronze.printSchema()

StatementMeta(, 6f7de58d-2e89-4403-9885-2359387de05c, 9, Finished, Available, Finished, False)

bronze_reviews: 400 rows
root
 |-- _feed_source: string (nullable = true)
 |-- generated_at_utc: string (nullable = true)
 |-- review: struct (nullable = true)
 |    |-- helpful_votes: long (nullable = true)
 |    |-- product_id: long (nullable = true)
 |    |-- rating: long (nullable = true)
 |    |-- review_date: string (nullable = true)
 |    |-- review_id: string (nullable = true)
 |    |-- review_text: string (nullable = true)
 |    |-- reviewer: struct (nullable = true)
 |    |    |-- location: struct (nullable = true)
 |    |    |    |-- city: string (nullable = true)
 |    |    |    |-- country: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- verified_purchase: boolean (nullable = true)
 |    |-- tags: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |-- _bronze_ingested_at: timestamp (nullable = true)
 |-- _source_system: string (nullable = false)
 |-- _source_file: string (nullable = false)



**JSON exchange rates source**

In [8]:
rates_raw = spark.read.option("multiLine", True).json("Files/raw/exchange_rates/*.json")

rates_bronze = add_audit_columns(rates_raw, "rest_api_exchange_rates")
rates_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_exchange_rates")
print(f"bronze_exchange_rates: {rates_bronze.count()} rows")

StatementMeta(, 6f7de58d-2e89-4403-9885-2359387de05c, 10, Finished, Available, Finished, False)

bronze_exchange_rates: 1 rows


**Validation summary**

In [9]:
tables = ["bronze_customers", "bronze_products", "bronze_orders", "bronze_order_items",
          "bronze_loyalty", "bronze_reviews", "bronze_exchange_rates"]

print("=" * 50)
print("BRONZE LAYER INGESTION SUMMARY")
print("=" * 50)
for t in tables:
    count = spark.table(t).count()
    print(f"{t:.<35}{count:>10} rows")

StatementMeta(, 6f7de58d-2e89-4403-9885-2359387de05c, 11, Finished, Available, Finished, False)

BRONZE LAYER INGESTION SUMMARY
bronze_customers...................       510 rows
bronze_products....................       100 rows
bronze_orders......................      2000 rows
bronze_order_items.................      6024 rows
bronze_loyalty.....................       305 rows
bronze_reviews.....................       400 rows
bronze_exchange_rates..............         1 rows
